In [ ]:
import pandas as pd
from rdkit import DataStructs
import time

# ─── File paths ──────────────────────────────────────────────────────────────
kiba_file    = '../dataset/KIBA_with_klifs_morgan_cleaned.csv'
vsds_file    = '../dataset/VSDS_truedecoy_integrated_data.csv'
spectra_file = '../dataset/SPECTRA_dataset/data_split_from_merged_no_smiles/band_0.10_0.50/train.csv'
output_overlap_file     = '../dataset/KIBA_SPECTRA_VSDS_10_50_from_merged_remove_overlap_data_t0.80_cleavage_spectra_seen_target.csv'
output_non_overlap_file = '../dataset/KIBA_SPECTRA_VSDS_10_50_from_merged_remove_overlap_data_t0.80_cleavage_spectra_unseen_target.csv'

TANIMOTO_THRESHOLD = 0.80   # Similarity threshold; adjust as needed

# ─── Step 5 filtering switches (disabled by default) ──────────────────────────
# Filter switch for the group whose targets overlap with SPECTRA (total > 100 and active > 10)
FILTER_OVERLAP_GROUP     = True
# Filter switch for the group whose targets do not overlap with SPECTRA (total > 100 and active > 10)
FILTER_NON_OVERLAP_GROUP = True

print("=" * 60)
print("Remove KIBA data overlapping with VSDS/SPECTRA and filter targets")
print("=" * 60)

# ─── Read data ────────────────────────────────────────────────────────────────
print("\nReading data files...")
kiba_df    = pd.read_csv(kiba_file)
vsds_df    = pd.read_csv(vsds_file)
spectra_df = pd.read_csv(spectra_file)

print(f"KIBA   original data : {len(kiba_df):>8,} records")
print(f"VSDS   data     : {len(vsds_df):>8,} records")
print(f"SPECTRA data    : {len(spectra_df):>8,} records")

# ─── Check required columns ───────────────────────────────────────────────────
print("\nChecking required columns...")
required_kiba_cols = ['Morgan', 'KLIFS', 'uniprot_id', 'KIBA']
missing_cols = [col for col in required_kiba_cols if col not in kiba_df.columns]
if missing_cols:
    print(f"Error: KIBAfile is missing columns: {missing_cols}")
    exit(1)

for name, df in [("VSDS", vsds_df), ("SPECTRA", spectra_df)]:
    for col in ['Morgan', 'KLIFS']:
        if col not in df.columns:
            print(f"Error: {name}file is missing columns '{col}'")
            exit(1)
print("✓ All required columns are present")


# ════════════════════════════════════════════════════════════════
# Utility function: convert the Morgan column (bit string / list string) to ExplicitBitVect
# ════════════════════════════════════════════════════════════════
def morgan_col_to_bitvects(series: pd.Series) -> list:
    """
    Convert plain binary strings (for example, '0010110...') to a list of RDKit ExplicitBitVect objects.
    Return None at positions where conversion fails.
    """
    fps = []
    for val in series:
        try:
            fps.append(DataStructs.CreateFromBitString(str(val).strip()))
        except Exception:
            fps.append(None)
    return fps


def per_target_stats(df: pd.DataFrame, label: str, apply_filter: bool) -> pd.DataFrame:
    """
    Calculate total, active, and inactive counts and proportions for each target by uniprot_id.
    When apply_filter=True, retain targets with total>100 and active>10.
    Return the statistics DataFrame and print a summary.
    """
    tmp = df.copy()
    tmp['is_active'] = tmp['KIBA'] <= 3

    stats = tmp.groupby('uniprot_id').agg(
        total_count  = ('is_active', 'count'),
        active_count = ('is_active', 'sum'),
    ).reset_index()

    stats['inactive_count']   = stats['total_count'] - stats['active_count']
    stats['active_pct']       = stats['active_count']   / stats['total_count'] * 100
    stats['inactive_pct']     = stats['inactive_count'] / stats['total_count'] * 100
    stats = stats.sort_values('total_count', ascending=False).reset_index(drop=True)

    total_records = stats['total_count'].sum()
    total_active  = stats['active_count'].sum()
    total_inactive = stats['inactive_count'].sum()

    print(f"\n  [{label}] targets: {len(stats):,}  "
          f"total records: {total_records:,}  "
          f"Active: {total_active:,} ({total_active/total_records*100:.2f}%)  "
          f"Inactive: {total_inactive:,} ({total_inactive/total_records*100:.2f}%)")
    print(f"  Top 10 targets by record count:")
    print(stats[['uniprot_id', 'total_count', 'active_count',
                 'inactive_count', 'active_pct', 'inactive_pct']].head(10).to_string(index=False))

    if apply_filter:
        before = len(stats)
        stats = stats[
            (stats['total_count'] > 100) &
            (stats['active_count'] > 10)
        ].reset_index(drop=True)
        print(f"\n  After filtering (total>100 and active>10): {before} → {len(stats)} targets")
        total_records = stats['total_count'].sum()
        total_active  = stats['active_count'].sum()
        total_inactive = stats['inactive_count'].sum()
        print(f"  Total records after filtering: {total_records:,}  "
              f"Active: {total_active:,} ({total_active/max(total_records,1)*100:.2f}%)  "
              f"Inactive: {total_inactive:,} ({total_inactive/max(total_records,1)*100:.2f}%)")
    else:
        print(f"\n  [Filtering is disabled; retaining all targets]")

    return stats


# ════════════════════════════════════════════════════════════════
# Step 1: Remove Morgan+KLIFS combinations identical to those in VSDS
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Step 1: Remove Morgan+KLIFS combinations identical to those in VSDS")
print("=" * 60)

vsds_combos = set(
    zip(vsds_df['Morgan'].astype(str), vsds_df['KLIFS'].astype(str))
)
print(f"VSDS unique Morgan-KLIFS combinations: {len(vsds_combos):,}")

kiba_df['_combo'] = list(zip(
    kiba_df['Morgan'].astype(str),
    kiba_df['KLIFS'].astype(str)
))
vsds_exact_mask  = kiba_df['_combo'].isin(vsds_combos)
vsds_exact_count = vsds_exact_mask.sum()

print(f"KIBA records exactly overlapping with VSDS: {vsds_exact_count:,}  "
      f"({vsds_exact_count / len(kiba_df) * 100:.2f}%)")

kiba_step1 = kiba_df[~vsds_exact_mask].copy()
kiba_step1 = kiba_step1.drop(columns=['_combo'])
print(f"Remaining after removal: {len(kiba_step1):,} records  "
      f"(retention rate {len(kiba_step1) / len(kiba_df) * 100:.2f}%)")


# ════════════════════════════════════════════════════════════════
# Step 2: BulkTanimoto — remove KIBA molecules whose similarity to any VSDS molecule exceeds the threshold
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"Step 2: BulkTanimoto filtering (threshold > {TANIMOTO_THRESHOLD})")
print("=" * 60)

# 2a. Get unique VSDS Morgan fingerprints (deduplicate to accelerate computation)
print("Building the unique VSDS Morgan fingerprint set...")
vsds_unique_morgan = vsds_df['Morgan'].astype(str).drop_duplicates()
print(f"  VSDS unique Morgan fingerprints: {len(vsds_unique_morgan):,}")

t0 = time.time()
vsds_fps = morgan_col_to_bitvects(vsds_unique_morgan)
vsds_fps = [fp for fp in vsds_fps if fp is not None]   # Filter out failed conversions
print(f"  successfully converted fingerprints: {len(vsds_fps):,}  elapsed: {time.time()-t0:.1f}s")

# 2b. Get unique KIBA Morgan fingerprints after Step 1
print("\nBuilding unique KIBA Morgan fingerprints...")
kiba_unique_morgan = kiba_step1['Morgan'].astype(str).drop_duplicates().reset_index(drop=True)
print(f"  KIBA unique Morgan fingerprints: {len(kiba_unique_morgan):,}")

t0 = time.time()
kiba_fps_list = morgan_col_to_bitvects(kiba_unique_morgan)
print(f"  successfully converted fingerprints: {sum(fp is not None for fp in kiba_fps_list):,}  "
      f"elapsed: {time.time()-t0:.1f}s")

# 2c. Run BulkTanimoto calculations and mark Morgan values for removal
print(f"\nCalculating Tanimoto similarities in bulk ( {len(kiba_fps_list):,} × {len(vsds_fps):,} pairs)...")
t0 = time.time()
similar_morgan_set = set()

for i, kfp in enumerate(kiba_fps_list):
    if kfp is None:
        continue
    sims = DataStructs.BulkTanimotoSimilarity(kfp, vsds_fps)
    if max(sims) > TANIMOTO_THRESHOLD:
        similar_morgan_set.add(kiba_unique_morgan.iloc[i])

elapsed = time.time() - t0
print(f"  Completed; elapsed: {elapsed:.1f}s")
print(f"  KIBA unique Morgan fingerprints to remove (max_sim > {TANIMOTO_THRESHOLD}): "
      f"{len(similar_morgan_set):,}")

# 2d. Remove highly similar rows from kiba_step1
tanimoto_mask  = kiba_step1['Morgan'].astype(str).isin(similar_morgan_set)
tanimoto_count = tanimoto_mask.sum()

print(f"  corresponding rows to remove: {tanimoto_count:,}  "
      f"({tanimoto_count / len(kiba_step1) * 100:.2f}%)")

kiba_step2 = kiba_step1[~tanimoto_mask].copy()
print(f"  Remaining after removal: {len(kiba_step2):,} records  "
      f"(retention rate relative to original data {len(kiba_step2) / len(kiba_df) * 100:.2f}%)")

# Summary of Steps 1 and 2
print(f"\n──────────────────────────────────────────────────────")
print(f"Step 1 (exact overlap) removed  : {vsds_exact_count:>6,} records")
print(f"Step 2 (high similarity) removed  : {tanimoto_count:>6,} records")
print(f"Total removed in Steps 1–2              : {len(kiba_df) - len(kiba_step2):>6,} records")
print(f"──────────────────────────────────────────────────────")


# ════════════════════════════════════════════════════════════════
# Step 3: Remove Morgan+KLIFS combinations identical to those in SPECTRA
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Step 3: Remove Morgan+KLIFS combinations identical to those in SPECTRA")
print("=" * 60)

spectra_combos = set(
    zip(spectra_df['Morgan'].astype(str), spectra_df['KLIFS'].astype(str))
)
print(f"SPECTRA unique Morgan-KLIFS combinations: {len(spectra_combos):,}")

kiba_step2['_combo'] = list(zip(
    kiba_step2['Morgan'].astype(str),
    kiba_step2['KLIFS'].astype(str)
))
spectra_exact_mask  = kiba_step2['_combo'].isin(spectra_combos)
spectra_exact_count = spectra_exact_mask.sum()

print(f"KIBA records exactly overlapping with SPECTRA: {spectra_exact_count:,}  "
      f"({spectra_exact_count / len(kiba_step2) * 100:.2f}%)")

kiba_step3 = kiba_step2[~spectra_exact_mask].copy()
kiba_step3 = kiba_step3.drop(columns=['_combo'])

print(f"Remaining after removal: {len(kiba_step3):,} records  "
      f"(retention rate relative to original data {len(kiba_step3) / len(kiba_df) * 100:.2f}%)")


# ════════════════════════════════════════════════════════════════
# Step 4: BulkTanimoto — remove KIBA molecules whose similarity to any SPECTRA molecule exceeds the threshold
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"Step 4: BulkTanimoto filtering against SPECTRA (threshold > {TANIMOTO_THRESHOLD})")
print("=" * 60)

# 4a. Get unique SPECTRA Morgan fingerprints (deduplicated)
print("Building the unique SPECTRA Morgan fingerprint set...")
spectra_unique_morgan = spectra_df['Morgan'].astype(str).drop_duplicates()
print(f"  SPECTRA unique Morgan fingerprints: {len(spectra_unique_morgan):,}")

t0 = time.time()
spectra_fps = morgan_col_to_bitvects(spectra_unique_morgan)
spectra_fps = [fp for fp in spectra_fps if fp is not None]
print(f"  successfully converted fingerprints: {len(spectra_fps):,}  elapsed: {time.time()-t0:.1f}s")

# 4b. Get unique KIBA Morgan fingerprints after Step 3
print("\nBuilding unique KIBA Morgan fingerprints...")
kiba_unique_morgan_s3 = kiba_step3['Morgan'].astype(str).drop_duplicates().reset_index(drop=True)
print(f"  KIBA unique Morgan fingerprints: {len(kiba_unique_morgan_s3):,}")

t0 = time.time()
kiba_fps_list_s3 = morgan_col_to_bitvects(kiba_unique_morgan_s3)
print(f"  successfully converted fingerprints: {sum(fp is not None for fp in kiba_fps_list_s3):,}  "
      f"elapsed: {time.time()-t0:.1f}s")

# 4c. Run BulkTanimoto calculations
print(f"\nCalculating Tanimoto similarities in bulk ( {len(kiba_fps_list_s3):,} × {len(spectra_fps):,} pairs)...")
t0 = time.time()
spectra_similar_morgan_set = set()

for i, kfp in enumerate(kiba_fps_list_s3):
    if kfp is None:
        continue
    sims = DataStructs.BulkTanimotoSimilarity(kfp, spectra_fps)
    if max(sims) > TANIMOTO_THRESHOLD:
        spectra_similar_morgan_set.add(kiba_unique_morgan_s3.iloc[i])

elapsed = time.time() - t0
print(f"  Completed; elapsed: {elapsed:.1f}s")
print(f"  KIBA unique Morgan fingerprints to remove (max_sim > {TANIMOTO_THRESHOLD}): "
      f"{len(spectra_similar_morgan_set):,}")

# 4d. Remove highly similar rows
spectra_tanimoto_mask  = kiba_step3['Morgan'].astype(str).isin(spectra_similar_morgan_set)
spectra_tanimoto_count = spectra_tanimoto_mask.sum()

print(f"  corresponding rows to remove: {spectra_tanimoto_count:,}  "
      f"({spectra_tanimoto_count / len(kiba_step3) * 100:.2f}%)")

kiba_step4 = kiba_step3[~spectra_tanimoto_mask].copy()
print(f"  Remaining after removal: {len(kiba_step4):,} records  "
      f"(retention rate relative to original data {len(kiba_step4) / len(kiba_df) * 100:.2f}%)")

# Summary of Steps 1–4
print(f"\n──────────────────────────────────────────────────────")
print(f"Step 1 VSDS exact-overlap removal  : {vsds_exact_count:>6,} records")
print(f"Step 2 VSDS high-similarity removal  : {tanimoto_count:>6,} records")
print(f"Step 3 SPECTRA exact-overlap removal: {spectra_exact_count:>6,} records")
print(f"Step 4 SPECTRA high-similarity removal: {spectra_tanimoto_count:>6,} records")
print(f"Total removed in Steps 1–4                 : {len(kiba_df) - len(kiba_step4):>6,} records")
print(f"──────────────────────────────────────────────────────")


# ════════════════════════════════════════════════════════════════
# Step 5: Group by target overlap with SPECTRA and calculate per-target statistics
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Step 5: Group and summarize by target overlap with SPECTRA")
print("=" * 60)

# Use the KLIFS column to determine whether targets overlap with SPECTRA
spectra_klifs_set = set(spectra_df['KLIFS'].dropna().astype(str))
kiba_step4['_in_spectra'] = kiba_step4['KLIFS'].astype(str).isin(spectra_klifs_set)

overlap_group     = kiba_step4[kiba_step4['_in_spectra']].drop(columns=['_in_spectra']).copy()
non_overlap_group = kiba_step4[~kiba_step4['_in_spectra']].drop(columns=['_in_spectra']).copy()

print(f"\nKIBA target (KLIFS) records overlapping with SPECTRA : {len(overlap_group):,}")
print(f"KIBA target (KLIFS) records not overlapping with SPECTRA   : {len(non_overlap_group):,}")

# ─── Overlapping-group statistics ─────────────────────────────────────────────
print("\n" + "─" * 60)
print(f"▶ Group with targets overlapping SPECTRA  "
      f"[FILTER_OVERLAP_GROUP = {FILTER_OVERLAP_GROUP}]")
print("─" * 60)
overlap_stats = per_target_stats(overlap_group, "overlapping group", FILTER_OVERLAP_GROUP)

# ─── Non-overlapping-group statistics ─────────────────────────────────────────
print("\n" + "─" * 60)
print(f"▶ Group with targets not overlapping SPECTRA  "
      f"[FILTER_NON_OVERLAP_GROUP = {FILTER_NON_OVERLAP_GROUP}]")
print("─" * 60)
non_overlap_stats = per_target_stats(non_overlap_group, "non-overlapping group", FILTER_NON_OVERLAP_GROUP)

# ─── If filtering is enabled, rebuild kiba_final using the filtered target sets ─
if FILTER_OVERLAP_GROUP or FILTER_NON_OVERLAP_GROUP:
    valid_targets_overlap     = set(overlap_stats['uniprot_id'])     if FILTER_OVERLAP_GROUP     else set(overlap_group['uniprot_id'])
    valid_targets_non_overlap = set(non_overlap_stats['uniprot_id']) if FILTER_NON_OVERLAP_GROUP else set(non_overlap_group['uniprot_id'])
    valid_targets_all = valid_targets_overlap | valid_targets_non_overlap
    kiba_final = kiba_step4[kiba_step4['uniprot_id'].isin(valid_targets_all)].copy()
    print(f"\nFinal data size after filtering: {len(kiba_final):,} records  "
          f"(retention rate relative to original data {len(kiba_final) / len(kiba_df) * 100:.2f}%)")
else:
    kiba_final = kiba_step4.copy()
    print(f"\n[Filtering is disabled for both groups] final data size: {len(kiba_final):,} records  "
          f"(retention rate relative to original data {len(kiba_final) / len(kiba_df) * 100:.2f}%)")


# ════════════════════════════════════════════════════════════════
# Step 6: Analyze target overlap among KIBA, VSDS, and SPECTRA based on KLIFS encoding
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Step 6: Analyze KIBA / VSDS / SPECTRA target overlap (KLIFS)")
print("=" * 60)

kiba_targets    = set(kiba_final['KLIFS'].dropna().astype(str))
vsds_targets    = set(vsds_df['KLIFS'].dropna().astype(str))
spectra_targets = set(spectra_df['KLIFS'].dropna().astype(str))

print(f"\nKIBA    unique targets (after filtering) : {len(kiba_targets):>5,}")
print(f"VSDS    unique targets          : {len(vsds_targets):>5,}")
print(f"SPECTRA unique targets          : {len(spectra_targets):>5,}")

# ─── Seven-region decomposition ────────────────────────────────────────────────
kv   = kiba_targets & vsds_targets
ks   = kiba_targets & spectra_targets
vs   = vsds_targets & spectra_targets
kvs  = kiba_targets & vsds_targets & spectra_targets

kiba_only    = kiba_targets    - vsds_targets - spectra_targets
vsds_only    = vsds_targets    - kiba_targets - spectra_targets
spectra_only = spectra_targets - kiba_targets - vsds_targets
kv_only      = kv  - spectra_targets
ks_only      = ks  - vsds_targets
vs_only      = vs  - kiba_targets

print(f"\n─── Intersection analysis ────────────────────────────────────────────")
print(f"KIBA ∩ VSDS               : {len(kv):>5,}  "
      f"(of KIBA {len(kv)/max(len(kiba_targets),1)*100:.1f}%  "
      f"of VSDS {len(kv)/max(len(vsds_targets),1)*100:.1f}%)")
print(f"KIBA ∩ SPECTRA            : {len(ks):>5,}  "
      f"(of KIBA {len(ks)/max(len(kiba_targets),1)*100:.1f}%  "
      f"of SPECTRA {len(ks)/max(len(spectra_targets),1)*100:.1f}%)")
print(f"VSDS ∩ SPECTRA            : {len(vs):>5,}  "
      f"(of VSDS {len(vs)/max(len(vsds_targets),1)*100:.1f}%  "
      f"of SPECTRA {len(vs)/max(len(spectra_targets),1)*100:.1f}%)")
print(f"KIBA ∩ VSDS ∩ SPECTRA     : {len(kvs):>5,}")

print(f"\n─── Exclusive targets ────────────────────────────────────────────")
print(f"KIBA only                  : {len(kiba_only):>5,}")
print(f"VSDS only                  : {len(vsds_only):>5,}")
print(f"SPECTRA only               : {len(spectra_only):>5,}")
print(f"KIBA∩VSDS（excluding SPECTRA）   : {len(kv_only):>5,}")
print(f"KIBA∩SPECTRA（excluding VSDS）   : {len(ks_only):>5,}")
print(f"VSDS∩SPECTRA（excluding KIBA）   : {len(vs_only):>5,}")

# ─── Jaccard similarity ────────────────────────────────────────────────────────
def jaccard(a, b):
    u = a | b
    return len(a & b) / len(u) if u else 0.0

print(f"\n─── Jaccard similarity ──────────────────────────────────────")
print(f"KIBA vs VSDS    : {jaccard(kiba_targets, vsds_targets):.4f}")
print(f"KIBA vs SPECTRA : {jaccard(kiba_targets, spectra_targets):.4f}")
print(f"VSDS vs SPECTRA : {jaccard(vsds_targets, spectra_targets):.4f}")

# ─── Targets shared by all three datasets ──────────────────────────────────────
if kvs:
    print(f"\n─── KIBA ∩ VSDS ∩ SPECTRA shared targets ({len(kvs)} targets) ────────")
    for t in sorted(kvs):
        print(f"  {t}")

# ─── Appendix: comparison with the original unfiltered KIBA data ────────────────
kiba_raw_targets = set(kiba_df['KLIFS'].dropna().astype(str))
kv_raw  = kiba_raw_targets & vsds_targets
ks_raw  = kiba_raw_targets & spectra_targets
kvs_raw = kiba_raw_targets & vsds_targets & spectra_targets

print(f"\n─── Target-overlap comparison with original KIBA (unfiltered) ─────────────────────")
print(f"Original KIBA unique targets        : {len(kiba_raw_targets):>5,}")
print(f"Original KIBA ∩ VSDS           : {len(kv_raw):>5,}  "
      f"({len(kv_raw)/max(len(kiba_raw_targets),1)*100:.1f}%)")
print(f"Original KIBA ∩ SPECTRA        : {len(ks_raw):>5,}  "
      f"({len(ks_raw)/max(len(kiba_raw_targets),1)*100:.1f}%)")
print(f"Original KIBA ∩ VSDS ∩ SPECTRA : {len(kvs_raw):>5,}")


# ─── Save results ──────────────────────────────────────────────────────────────
print(f"\nSaving results...")

# If filtering is enabled, kiba_final is already filtered; regroup it before saving
final_in_spectra = kiba_final['KLIFS'].astype(str).isin(spectra_klifs_set)
final_overlap     = kiba_final[final_in_spectra].copy()
final_non_overlap = kiba_final[~final_in_spectra].copy()

final_overlap.to_csv(output_overlap_file, index=False)
print(f"✓ Group with targets overlapping SPECTRA   → {output_overlap_file}")
print(f"  records: {len(final_overlap):,}  unique targets: {final_overlap['uniprot_id'].nunique():,}")

final_non_overlap.to_csv(output_non_overlap_file, index=False)
print(f"✓ Group with targets not overlapping SPECTRA → {output_non_overlap_file}")
print(f"  records: {len(final_non_overlap):,}  unique targets: {final_non_overlap['uniprot_id'].nunique():,}")

print("\n" + "=" * 60)
print("✅ All tasks completed")
print("=" * 60)